<a href="https://colab.research.google.com/github/juvedibuar00/Artigo---dht---inmet---conv/blob/main/dados_faltantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# IMPORTAÇÕES
# ============================================================
import pandas as pd
import numpy as np
from google.colab import files   # se estiver no Colab

# ============================================================
# 1. UPLOAD DO ARQUIVO (Google Colab)
# ============================================================
print("📤 Faça o upload do arquivo 'ESTACAO_DADOS_CONVENCIONAIS.csv'")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Ler o CSV (o pandas lida com cabeçalhos duplicados adicionando sufixos .1, .2, ...)
df = pd.read_csv(filename, encoding='utf-8')

# ============================================================
# 2. RENOMEAR COLUNAS PARA NOMES CLAROS
# ============================================================
# O cabeçalho original: Data, 9h, 15h, 21h, media, 9h, 15h, 21h, media
# Após a leitura, as colunas ficam:
# Data, 9h, 15h, 21h, media, 9h.1, 15h.1, 21h.1, media.1
# Vamos renomear explicitamente:
df.columns = [
    'Data',
    'Ts9', 'Ts15', 'Ts21', 'Media_Temp',
    'Ur9', 'Ur15', 'Ur21', 'Media_UR'
]

# ============================================================
# 3. PRÉ-PROCESSAMENTO
# ============================================================
# Converter a coluna 'Data' para datetime
df['Data'] = pd.to_datetime(df['Data'], errors='coerce', format='%Y/%m/%d')

# Substituir strings vazias ou espaços por NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# Garantir que todas as colunas numéricas sejam float
cols = ['Ts9', 'Ts15', 'Ts21', 'Media_Temp', 'Ur9', 'Ur15', 'Ur21', 'Media_UR']
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ============================================================
# 4. PREENCHER Ts21 E Ur21 A PARTIR DAS MÉDIAS DIÁRIAS
# ============================================================
# Preencher Ts21 se houver Media_Temp e Ts9 e Ts15 existirem
mask_temp = df['Ts21'].isna() & df['Media_Temp'].notna()
df.loc[mask_temp, 'Ts21'] = (
    3 * df.loc[mask_temp, 'Media_Temp']
    - df.loc[mask_temp, 'Ts9']
    - df.loc[mask_temp, 'Ts15']
)

# Preencher Ur21 se houver Media_UR e Ur9 e Ur15 existirem
mask_ur = df['Ur21'].isna() & df['Media_UR'].notna()
df.loc[mask_ur, 'Ur21'] = (
    3 * df.loc[mask_ur, 'Media_UR']
    - df.loc[mask_ur, 'Ur9']
    - df.loc[mask_ur, 'Ur15']
)

# ============================================================
# 5. INTERPOLAÇÃO PARA DIAS COMPLETAMENTE AUSENTES
# ============================================================
# Colunas a serem interpoladas (todas as seis medições)
cols_to_interp = ['Ts9', 'Ts15', 'Ts21', 'Ur9', 'Ur15', 'Ur21']

# Identificar dias em que todas as seis colunas estão vazias
df['all_missing'] = df[cols_to_interp].isna().all(axis=1)

# Ordenar por data (garantir ordem cronológica)
df = df.sort_values('Data').reset_index(drop=True)

# Aplicar interpolação linear em cada coluna (preenche para frente e para trás)
for col in cols_to_interp:
    df[col] = df[col].interpolate(method='linear', limit_direction='both')

# ============================================================
# 6. RECALCULAR AS MÉDIAS DIÁRIAS
# ============================================================
df['Media_Temp'] = (df['Ts9'] + df['Ts15'] + df['Ts21']) / 3
df['Media_UR']   = (df['Ur9'] + df['Ur15'] + df['Ur21']) / 3

# ============================================================
# 7. REMOVER COLUNA AUXILIAR E EXPORTAR
# ============================================================
df.drop(columns=['all_missing'], inplace=True)

# (Opcional) Voltar ao formato original de data (se preferir)
# df['Data'] = df['Data'].dt.strftime('%Y/%m/%d')

# Salvar como CSV
output_filename = 'ESTACAO_DADOS_CONVENCIONAIS_PREENCHIDO.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig', float_format='%.1f')

# Baixar o arquivo (Colab)
files.download(output_filename)

print(f"✅ Arquivo salvo como '{output_filename}' e baixado com sucesso!")

📤 Faça o upload do arquivo 'ESTACAO_DADOS_CONVENCIONAIS.csv'


Saving ESTACAO_DADOS_CONVENCIONAIS.csv to ESTACAO_DADOS_CONVENCIONAIS (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Arquivo salvo como 'ESTACAO_DADOS_CONVENCIONAIS_PREENCHIDO.csv' e baixado com sucesso!
